# Prepare LoRA Data

Convert benchmark records into the chat JSONL format expected by `mlx_lm.lora`.

In [1]:
from pathlib import Path
import json
import sys

In [2]:
PROJECT_ROOT = Path("/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune")
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune')

In [3]:
from training_eval.eval_utils import load_jsonl_records

In [4]:
SYSTEM_MESSAGE = "You solve discrete stochastic-process problems. Give the reasoning, then put the final JSON answer inside <answer>...</answer>."

In [5]:
DATASETS = {
    "explicit_theorems": {
        "train": PROJECT_ROOT / "benchmark" / "data" / "train",
        "valid": PROJECT_ROOT / "benchmark" / "data" / "val",
    },
    "implicit_theorems": {
        "train": PROJECT_ROOT / "benchmark" / "data" / "train_implicit_theorems",
        "valid": PROJECT_ROOT / "benchmark" / "data" / "val_implicit_theorems",
    },
}

OUTPUT_ROOT = PROJECT_ROOT / "training_eval" / "fine_tune_qwen1_7B" / "lora" / "data"

In [6]:
def make_chat_example(record):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": record["problem"]},
            {"role": "assistant", "content": record["reasoning"]},
        ]
    }

In [7]:
def write_jsonl(records, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        for record in records:
            f.write(json.dumps(record, sort_keys=True) + "\n")

In [8]:
def build_lora_dataset(name, paths):
    train_records = load_jsonl_records(paths["train"])
    valid_records = load_jsonl_records(paths["valid"])
    output_dir = OUTPUT_ROOT / name

    write_jsonl([make_chat_example(record) for record in train_records], output_dir / "train.jsonl")
    write_jsonl([make_chat_example(record) for record in valid_records], output_dir / "valid.jsonl")

    return {
        "name": name,
        "output_dir": output_dir,
        "train_records": len(train_records),
        "valid_records": len(valid_records),
    }

In [9]:
summaries = [build_lora_dataset(name, paths) for name, paths in DATASETS.items()]
summaries

[{'name': 'explicit_theorems',
  'output_dir': PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/training_eval/fine_tune_qwen1_7B/lora/data/explicit_theorems'),
  'train_records': 480,
  'valid_records': 240},
 {'name': 'implicit_theorems',
  'output_dir': PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/training_eval/fine_tune_qwen1_7B/lora/data/implicit_theorems'),
  'train_records': 480,
  'valid_records': 240}]

In [10]:
example_path = OUTPUT_ROOT / "explicit_theorems" / "train.jsonl"
json.loads(example_path.read_text().splitlines()[0])

{'messages': [{'content': 'You solve discrete stochastic-process problems. Give the reasoning, then put the final JSON answer inside <answer>...</answer>.',
   'role': 'system'},
  {'content': 'Consider a simple symmetric random walk (X_n) on the integers with X_0 = 8. Set tau = inf{n >= 0 : X_n in {0, 10}}. Compute the expected hitting time E[tau]. Return JSON of the form {"expected_time": "..."} inside the answer tags.',
   'role': 'user'},
  {'content': 'For this finite-boundary stopped walk, use the bounded optional-stopping principle: if (M_n) is a martingale and the stopped family (M_{n wedge tau}) is bounded, then E[M_tau] = E[M_0] for the finite-valued limit. Use the usual pair of martingales for the finite interval walk. First, (X_n) is a martingale, and optional stopping gives E[X_tau] = 8. Use the quadratic martingale for centered independent increments: if S_n = S_0 + Y_1 + ... + Y_n, the increments have mean 0 and variance sigma^2, and the increments are independent of the